# 01 — Common PPG-DaLiA Native-Rate Preprocessing
Raw data is read from:

```text
/home/iailab42/g2-synthetic/PPG_FieldStudy/
```

Project outputs are saved inside:

```text
/home/iailab42/khans1/projects/ir/
```

```text
data/processed/native_rates/
```


In [ ]:

# ============================================================
# 01_preprocessing_common_native_rates.py
#
# Common native-rate preprocessing for PPG-DaLiA wrist signals.
#
# This version follows the group preprocessing format and saves
# the exact compatibility files expected by the existing evaluation
# notebooks:
#
#   processed_all_subjects_native_rates/
#       all_X_acc_32hz.npy
#       all_X_bvp_64hz.npy
#       all_X_slow_4hz.npy
#       all_y.npy
#       all_subject.npy
#       all_metadata.csv
#       normalization_stats_native.npz
#
# It also saves the same outputs inside the clean project structure:
#
#   data/processed/native_rates/
#   configs/
#   figures/preprocessing/
#   logs/
#   results/preprocessing/
#   discussion/
#
# Raw data:
#   /home/iailab42/g2-synthetic/PPG_FieldStudy/
#
# Project:
#   /home/iailab42/khans1/projects/ir/
# ============================================================

from __future__ import annotations

import json
import logging
import pickle
import random
import shutil
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

@dataclass
class PreprocessingConfig:
    project_root: str = "/home/iailab42/khans1/projects/ir"
    raw_data_dir: str = "/home/iailab42/g2-synthetic/PPG_FieldStudy"

    subject_ids: List[int] = field(default_factory=lambda: list(range(1, 16)))
    random_seed: int = 42

    acc_hz: int = 32
    bvp_hz: int = 64
    slow_hz: int = 4
    label_hz: int = 4

    window_seconds: float = 8.0
    shift_seconds: float = 2.0

    min_label_coverage: float = 0.80
    drop_activity_zero: bool = True

    save_raw_arrays: bool = True
    save_debug_csvs: bool = True
    num_debug_windows: int = 10

    # Keep the same split protocol used in the provided evaluation notebooks.
    # This notebook does not create split arrays; it only saves this protocol.
    train_subjects: List[str] = field(default_factory=lambda: [
        "S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"
    ])
    val_subjects: List[str] = field(default_factory=lambda: ["S14", "S15"])
    test_subjects: List[str] = field(default_factory=lambda: ["S7", "S8", "S10"])

    # PPG-DaLiA S6 hardware issue guard.
    s6_max_valid_seconds: int = 90 * 60

    acc_channel_names: List[str] = field(default_factory=lambda: ["ACC_x", "ACC_y", "ACC_z"])
    bvp_channel_names: List[str] = field(default_factory=lambda: ["BVP"])
    slow_channel_names: List[str] = field(default_factory=lambda: ["EDA", "TEMP"])

    activity_names: Dict[int, str] = field(default_factory=lambda: {
        0: "transient_or_unlabeled",
        1: "sitting",
        2: "stairs",
        3: "soccer",
        4: "cycling",
        5: "driving",
        6: "lunch",
        7: "walking",
        8: "working",
    })


CONFIG = PreprocessingConfig()


# ============================================================
# Paths, logging, reproducibility
# ============================================================

def get_project_paths(config: PreprocessingConfig) -> Dict[str, Path]:
    project_root = Path(config.project_root)

    return {
        "project_root": project_root,
        "configs": project_root / "configs",
        "data": project_root / "data",
        "processed": project_root / "data" / "processed" / "native_rates",
        "discussion": project_root / "discussion",
        "figures": project_root / "figures" / "preprocessing",
        "logs": project_root / "logs",
        "models": project_root / "models",
        "notebooks": project_root / "notebooks",
        "results": project_root / "results" / "preprocessing",
    }


def create_project_dirs(paths: Dict[str, Path]) -> None:
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)


def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def setup_logging(log_path: Path) -> logging.Logger:
    logger = logging.getLogger("common_ppg_dalia_preprocessing")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_path, mode="w")
    file_handler.setFormatter(formatter)
    file_handler.setLevel(logging.INFO)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    stream_handler.setLevel(logging.INFO)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

    return logger


def save_json(data: dict, path: Path) -> None:
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def save_config_files(config: PreprocessingConfig, paths: Dict[str, Path]) -> None:
    save_json(asdict(config), paths["configs"] / "preprocessing_config.json")

    shared_split = {
        "description": "Shared subject split protocol used by KoVAE, GAN, Diffusion, and downstream evaluation.",
        "train_subjects": config.train_subjects,
        "val_subjects": config.val_subjects,
        "test_subjects": config.test_subjects,
    }
    save_json(shared_split, paths["configs"] / "shared_subject_split.json")


# ============================================================
# Basic signal helpers
# ============================================================

def load_pkl(path: Path) -> dict:
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")


def ensure_2d(signal: np.ndarray) -> np.ndarray:
    signal = np.asarray(signal, dtype=np.float32)

    if signal.ndim == 1:
        signal = signal[:, None]

    if signal.ndim != 2:
        raise ValueError(f"Expected 1D or 2D signal, got shape {signal.shape}")

    return signal


def crop_to_seconds(signal: np.ndarray, sampling_rate_hz: int, duration_sec: float) -> np.ndarray:
    n_samples = int(np.floor(duration_sec * sampling_rate_hz))
    return signal[:n_samples]


def sample_index(time_sec: float, sampling_rate_hz: int) -> int:
    return int(round(time_sec * sampling_rate_hz))


def compute_window_lengths(config: PreprocessingConfig) -> Dict[str, int]:
    return {
        "acc_window_len": int(config.window_seconds * config.acc_hz),
        "bvp_window_len": int(config.window_seconds * config.bvp_hz),
        "slow_window_len": int(config.window_seconds * config.slow_hz),
        "label_window_len": int(config.window_seconds * config.label_hz),
        "acc_shift_len": int(config.shift_seconds * config.acc_hz),
        "bvp_shift_len": int(config.shift_seconds * config.bvp_hz),
        "slow_shift_len": int(config.shift_seconds * config.slow_hz),
        "label_shift_len": int(config.shift_seconds * config.label_hz),
    }


# ============================================================
# Windowing
# ============================================================

def make_native_rate_windows(
    acc: np.ndarray,
    bvp: np.ndarray,
    eda: np.ndarray,
    temp: np.ndarray,
    activity: np.ndarray,
    duration_sec: float,
    config: PreprocessingConfig,
) -> Optional[Dict[str, np.ndarray]]:
    lengths = compute_window_lengths(config)

    if duration_sec < config.window_seconds:
        return None

    num_windows_total = int(np.floor((duration_sec - config.window_seconds) / config.shift_seconds)) + 1

    X_acc_list = []
    X_bvp_list = []
    X_slow_list = []
    y_list = []
    coverage_list = []

    start_time_list = []
    end_time_list = []
    acc_start_list = []
    acc_end_list = []
    bvp_start_list = []
    bvp_end_list = []
    slow_start_list = []
    slow_end_list = []
    label_start_list = []
    label_end_list = []

    for win_idx in range(num_windows_total):
        start_time_sec = win_idx * config.shift_seconds
        end_time_sec = start_time_sec + config.window_seconds

        acc_start = sample_index(start_time_sec, config.acc_hz)
        acc_end = acc_start + lengths["acc_window_len"]

        bvp_start = sample_index(start_time_sec, config.bvp_hz)
        bvp_end = bvp_start + lengths["bvp_window_len"]

        slow_start = sample_index(start_time_sec, config.slow_hz)
        slow_end = slow_start + lengths["slow_window_len"]

        label_start = sample_index(start_time_sec, config.label_hz)
        label_end = label_start + lengths["label_window_len"]

        if acc_end > len(acc):
            continue
        if bvp_end > len(bvp):
            continue
        if slow_end > len(eda) or slow_end > len(temp):
            continue
        if label_end > len(activity):
            continue

        acc_win = acc[acc_start:acc_end]
        bvp_win = bvp[bvp_start:bvp_end]

        eda_win = eda[slow_start:slow_end]
        temp_win = temp[slow_start:slow_end]
        slow_win = np.concatenate([eda_win, temp_win], axis=1)

        activity_win = activity[label_start:label_end]
        labels, counts = np.unique(activity_win, return_counts=True)

        dominant_index = int(np.argmax(counts))
        dominant_label = int(labels[dominant_index])
        coverage = counts[dominant_index] / float(lengths["label_window_len"])

        if config.drop_activity_zero and dominant_label == 0:
            continue

        if coverage < config.min_label_coverage:
            continue

        if not np.all(np.isfinite(acc_win)):
            continue
        if not np.all(np.isfinite(bvp_win)):
            continue
        if not np.all(np.isfinite(slow_win)):
            continue

        X_acc_list.append(acc_win.astype(np.float32))
        X_bvp_list.append(bvp_win.astype(np.float32))
        X_slow_list.append(slow_win.astype(np.float32))

        y_list.append(dominant_label)
        coverage_list.append(float(coverage))

        start_time_list.append(float(start_time_sec))
        end_time_list.append(float(end_time_sec))

        acc_start_list.append(int(acc_start))
        acc_end_list.append(int(acc_end))
        bvp_start_list.append(int(bvp_start))
        bvp_end_list.append(int(bvp_end))
        slow_start_list.append(int(slow_start))
        slow_end_list.append(int(slow_end))
        label_start_list.append(int(label_start))
        label_end_list.append(int(label_end))

    if len(y_list) == 0:
        return None

    return {
        "X_acc": np.stack(X_acc_list).astype(np.float32),
        "X_bvp": np.stack(X_bvp_list).astype(np.float32),
        "X_slow": np.stack(X_slow_list).astype(np.float32),
        "y": np.asarray(y_list, dtype=np.int64),
        "coverage": np.asarray(coverage_list, dtype=np.float32),
        "start_time_sec": np.asarray(start_time_list, dtype=np.float32),
        "end_time_sec": np.asarray(end_time_list, dtype=np.float32),
        "acc_start": np.asarray(acc_start_list, dtype=np.int64),
        "acc_end": np.asarray(acc_end_list, dtype=np.int64),
        "bvp_start": np.asarray(bvp_start_list, dtype=np.int64),
        "bvp_end": np.asarray(bvp_end_list, dtype=np.int64),
        "slow_start": np.asarray(slow_start_list, dtype=np.int64),
        "slow_end": np.asarray(slow_end_list, dtype=np.int64),
        "label_start": np.asarray(label_start_list, dtype=np.int64),
        "label_end": np.asarray(label_end_list, dtype=np.int64),
    }


# ============================================================
# Subject processing
# ============================================================

def process_subject(
    subject_id: int,
    config: PreprocessingConfig,
    logger: logging.Logger,
) -> Optional[Dict[str, object]]:
    subject_name = f"S{subject_id}"
    pkl_path = Path(config.raw_data_dir) / subject_name / f"{subject_name}.pkl"

    if not pkl_path.exists():
        logger.warning("Skipping %s because file was not found: %s", subject_name, pkl_path)
        return None

    logger.info("=" * 70)
    logger.info("Processing %s", subject_name)
    logger.info("File: %s", pkl_path)

    data = load_pkl(pkl_path)
    wrist = data["signal"]["wrist"]

    acc = ensure_2d(wrist["ACC"])
    bvp = ensure_2d(wrist["BVP"])
    eda = ensure_2d(wrist["EDA"])
    temp = ensure_2d(wrist["TEMP"])
    activity = np.asarray(data["activity"]).reshape(-1).astype(np.int64)

    duration_acc = len(acc) / float(config.acc_hz)
    duration_bvp = len(bvp) / float(config.bvp_hz)
    duration_eda = len(eda) / float(config.slow_hz)
    duration_temp = len(temp) / float(config.slow_hz)
    duration_activity = len(activity) / float(config.label_hz)

    duration_before_guard = min(
        duration_acc,
        duration_bvp,
        duration_eda,
        duration_temp,
        duration_activity,
    )

    duration_sec = float(duration_before_guard)
    s6_special_crop_applied = False

    if subject_name == "S6" and duration_sec > config.s6_max_valid_seconds:
        duration_sec = float(config.s6_max_valid_seconds)
        s6_special_crop_applied = True
        logger.info("Applied S6 guard: cropped to %.2f seconds", duration_sec)

    acc = crop_to_seconds(acc, config.acc_hz, duration_sec)
    bvp = crop_to_seconds(bvp, config.bvp_hz, duration_sec)
    eda = crop_to_seconds(eda, config.slow_hz, duration_sec)
    temp = crop_to_seconds(temp, config.slow_hz, duration_sec)
    activity = crop_to_seconds(activity, config.label_hz, duration_sec)

    result = make_native_rate_windows(
        acc=acc,
        bvp=bvp,
        eda=eda,
        temp=temp,
        activity=activity,
        duration_sec=duration_sec,
        config=config,
    )

    if result is None:
        logger.warning("%s produced no valid windows after filtering.", subject_name)
        return None

    n_windows = len(result["y"])
    subjects = np.array([subject_name] * n_windows, dtype=object)

    logger.info("Windowed %s:", subject_name)
    logger.info("  X_acc:  %s", result["X_acc"].shape)
    logger.info("  X_bvp:  %s", result["X_bvp"].shape)
    logger.info("  X_slow: %s", result["X_slow"].shape)
    logger.info("  y:      %s", result["y"].shape)
    logger.info("  label counts: %s", dict(zip(*np.unique(result["y"], return_counts=True))))

    result["subjects"] = subjects
    result["summary"] = {
        "subject": subject_name,
        "duration_acc_sec_original": float(duration_acc),
        "duration_bvp_sec_original": float(duration_bvp),
        "duration_eda_sec_original": float(duration_eda),
        "duration_temp_sec_original": float(duration_temp),
        "duration_activity_sec_original": float(duration_activity),
        "min_duration_before_s6_guard_sec": float(duration_before_guard),
        "duration_after_s6_guard_sec": float(duration_sec),
        "s6_special_crop_applied": bool(s6_special_crop_applied),
        "num_windows_after_filtering": int(n_windows),
    }

    return result


# ============================================================
# Normalization
# ============================================================

def compute_mean_std(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = X.mean(axis=(0, 1), keepdims=True).astype(np.float32)
    std = X.std(axis=(0, 1), keepdims=True).astype(np.float32)
    std = np.maximum(std, 1e-8).astype(np.float32)
    return mean, std


def normalize(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((X - mean) / std).astype(np.float32)


def make_normalization_summary(
    channel_names: List[str],
    mean: np.ndarray,
    std: np.ndarray,
    X_norm: np.ndarray,
) -> List[Dict[str, float]]:
    rows = []
    norm_mean = X_norm.mean(axis=(0, 1))
    norm_std = X_norm.std(axis=(0, 1))

    for name, raw_mean, raw_std, check_mean, check_std in zip(
        channel_names,
        mean.reshape(-1),
        std.reshape(-1),
        norm_mean.reshape(-1),
        norm_std.reshape(-1),
    ):
        rows.append({
            "channel": name,
            "raw_mean": float(raw_mean),
            "raw_std": float(raw_std),
            "normalized_mean_check": float(check_mean),
            "normalized_std_check": float(check_std),
        })

    return rows


# ============================================================
# Saving
# ============================================================

def save_core_outputs(
    output_dir: Path,
    config: PreprocessingConfig,
    X_acc_raw: np.ndarray,
    X_bvp_raw: np.ndarray,
    X_slow_raw: np.ndarray,
    X_acc_norm: np.ndarray,
    X_bvp_norm: np.ndarray,
    X_slow_norm: np.ndarray,
    y_all: np.ndarray,
    subjects_all: np.ndarray,
    coverage_all: np.ndarray,
    metadata_df: pd.DataFrame,
    subject_summary_df: pd.DataFrame,
    acc_mean: np.ndarray,
    acc_std: np.ndarray,
    bvp_mean: np.ndarray,
    bvp_std: np.ndarray,
    slow_mean: np.ndarray,
    slow_std: np.ndarray,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    np.save(output_dir / "all_X_acc_32hz.npy", X_acc_norm)
    np.save(output_dir / "all_X_bvp_64hz.npy", X_bvp_norm)
    np.save(output_dir / "all_X_slow_4hz.npy", X_slow_norm)
    np.save(output_dir / "all_y.npy", y_all)
    np.save(output_dir / "all_subject.npy", subjects_all)
    np.save(output_dir / "all_coverage.npy", coverage_all)

    np.savez_compressed(
        output_dir / "all_native_rate_arrays_normalized.npz",
        X_acc_32hz=X_acc_norm,
        X_bvp_64hz=X_bvp_norm,
        X_slow_4hz=X_slow_norm,
        y=y_all,
        subject=subjects_all,
        coverage=coverage_all,
    )

    if config.save_raw_arrays:
        np.save(output_dir / "all_X_acc_raw_32hz.npy", X_acc_raw)
        np.save(output_dir / "all_X_bvp_raw_64hz.npy", X_bvp_raw)
        np.save(output_dir / "all_X_slow_raw_4hz.npy", X_slow_raw)

        np.savez_compressed(
            output_dir / "all_native_rate_arrays_raw.npz",
            X_acc_raw_32hz=X_acc_raw,
            X_bvp_raw_64hz=X_bvp_raw,
            X_slow_raw_4hz=X_slow_raw,
            y=y_all,
            subject=subjects_all,
            coverage=coverage_all,
        )

    lengths = compute_window_lengths(config)

    np.savez(
        output_dir / "normalization_stats_native.npz",
        acc_mean=acc_mean.astype(np.float32),
        acc_std=acc_std.astype(np.float32),
        bvp_mean=bvp_mean.astype(np.float32),
        bvp_std=bvp_std.astype(np.float32),
        slow_mean=slow_mean.astype(np.float32),
        slow_std=slow_std.astype(np.float32),
        acc_channel_names=np.array(config.acc_channel_names),
        bvp_channel_names=np.array(config.bvp_channel_names),
        slow_channel_names=np.array(config.slow_channel_names),
        acc_hz=np.array([config.acc_hz], dtype=np.int64),
        bvp_hz=np.array([config.bvp_hz], dtype=np.int64),
        slow_hz=np.array([config.slow_hz], dtype=np.int64),
        label_hz=np.array([config.label_hz], dtype=np.int64),
        window_seconds=np.array([config.window_seconds], dtype=np.float32),
        shift_seconds=np.array([config.shift_seconds], dtype=np.float32),
        acc_window_len=np.array([lengths["acc_window_len"]], dtype=np.int64),
        bvp_window_len=np.array([lengths["bvp_window_len"]], dtype=np.int64),
        slow_window_len=np.array([lengths["slow_window_len"]], dtype=np.int64),
        label_window_len=np.array([lengths["label_window_len"]], dtype=np.int64),
        acc_shift_len=np.array([lengths["acc_shift_len"]], dtype=np.int64),
        bvp_shift_len=np.array([lengths["bvp_shift_len"]], dtype=np.int64),
        slow_shift_len=np.array([lengths["slow_shift_len"]], dtype=np.int64),
        label_shift_len=np.array([lengths["label_shift_len"]], dtype=np.int64),
        min_label_coverage=np.array([config.min_label_coverage], dtype=np.float32),
        drop_activity_zero=np.array([config.drop_activity_zero]),
        s6_max_valid_seconds=np.array([config.s6_max_valid_seconds], dtype=np.int64),
    )

    metadata_df.to_csv(output_dir / "all_metadata.csv", index=False)
    subject_summary_df.to_csv(output_dir / "subject_summary.csv", index=False)


def save_debug_csvs(
    output_dir: Path,
    config: PreprocessingConfig,
    X_acc_norm: np.ndarray,
    X_bvp_norm: np.ndarray,
    X_slow_norm: np.ndarray,
    y: np.ndarray,
    subjects: np.ndarray,
    coverage: np.ndarray,
    start_time_sec: np.ndarray,
) -> None:
    debug_dir = output_dir / "debug_csvs"
    debug_dir.mkdir(parents=True, exist_ok=True)

    n_debug = min(config.num_debug_windows, len(y))

    for i in range(n_debug):
        subject = str(subjects[i])
        label = int(y[i])
        start_sec = float(start_time_sec[i])

        df_acc = pd.DataFrame(X_acc_norm[i], columns=config.acc_channel_names)
        df_acc.insert(0, "time_step_acc_32hz", np.arange(X_acc_norm.shape[1]))
        df_acc.insert(1, "time_sec_relative", np.arange(X_acc_norm.shape[1]) / float(config.acc_hz))
        df_acc["activity_label"] = label
        df_acc["subject"] = subject
        df_acc["dominant_label_coverage"] = float(coverage[i])
        df_acc["window_start_time_sec"] = start_sec
        df_acc.to_csv(debug_dir / f"window_{i:04d}_{subject}_label_{label}_ACC_32hz.csv", index=False)

        df_bvp = pd.DataFrame(X_bvp_norm[i], columns=config.bvp_channel_names)
        df_bvp.insert(0, "time_step_bvp_64hz", np.arange(X_bvp_norm.shape[1]))
        df_bvp.insert(1, "time_sec_relative", np.arange(X_bvp_norm.shape[1]) / float(config.bvp_hz))
        df_bvp["activity_label"] = label
        df_bvp["subject"] = subject
        df_bvp["dominant_label_coverage"] = float(coverage[i])
        df_bvp["window_start_time_sec"] = start_sec
        df_bvp.to_csv(debug_dir / f"window_{i:04d}_{subject}_label_{label}_BVP_64hz.csv", index=False)

        df_slow = pd.DataFrame(X_slow_norm[i], columns=config.slow_channel_names)
        df_slow.insert(0, "time_step_slow_4hz", np.arange(X_slow_norm.shape[1]))
        df_slow.insert(1, "time_sec_relative", np.arange(X_slow_norm.shape[1]) / float(config.slow_hz))
        df_slow["activity_label"] = label
        df_slow["subject"] = subject
        df_slow["dominant_label_coverage"] = float(coverage[i])
        df_slow["window_start_time_sec"] = start_sec
        df_slow.to_csv(debug_dir / f"window_{i:04d}_{subject}_label_{label}_EDA_TEMP_4hz.csv", index=False)


# ============================================================
# Figures
# ============================================================

def plot_activity_distribution(y: np.ndarray, config: PreprocessingConfig, figure_path: Path) -> None:
    labels, counts = np.unique(y, return_counts=True)
    label_names = [config.activity_names.get(int(label), str(label)) for label in labels]

    plt.figure(figsize=(10, 5))
    plt.bar(label_names, counts)
    plt.xticks(rotation=45, ha="right")
    plt.xlabel("Activity")
    plt.ylabel("Number of windows")
    plt.title("Activity distribution after preprocessing")
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def plot_subject_window_counts(subjects: np.ndarray, figure_path: Path) -> None:
    subject_labels, counts = np.unique(subjects.astype(str), return_counts=True)
    order = np.argsort([int(s[1:]) for s in subject_labels])
    subject_labels = subject_labels[order]
    counts = counts[order]

    plt.figure(figsize=(10, 5))
    plt.bar(subject_labels, counts)
    plt.xlabel("Subject")
    plt.ylabel("Number of windows")
    plt.title("Window count per subject")
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def plot_example_window(
    X_acc: np.ndarray,
    X_bvp: np.ndarray,
    X_slow: np.ndarray,
    y: np.ndarray,
    subjects: np.ndarray,
    config: PreprocessingConfig,
    figure_path: Path,
) -> None:
    if len(y) == 0:
        return

    idx = 0
    activity = config.activity_names.get(int(y[idx]), str(int(y[idx])))
    subject = str(subjects[idx])

    fig, axes = plt.subplots(3, 1, figsize=(12, 8))

    t_bvp = np.arange(X_bvp.shape[1]) / config.bvp_hz
    axes[0].plot(t_bvp, X_bvp[idx, :, 0])
    axes[0].set_title(f"BVP 64 Hz | {subject} | {activity}")
    axes[0].set_xlabel("Time (sec)")
    axes[0].set_ylabel("Norm. BVP")

    t_acc = np.arange(X_acc.shape[1]) / config.acc_hz
    axes[1].plot(t_acc, X_acc[idx, :, 0], label="ACC_x")
    axes[1].plot(t_acc, X_acc[idx, :, 1], label="ACC_y")
    axes[1].plot(t_acc, X_acc[idx, :, 2], label="ACC_z")
    axes[1].set_title("ACC 32 Hz")
    axes[1].set_xlabel("Time (sec)")
    axes[1].set_ylabel("Norm. ACC")
    axes[1].legend(loc="best")

    t_slow = np.arange(X_slow.shape[1]) / config.slow_hz
    axes[2].plot(t_slow, X_slow[idx, :, 0], label="EDA")
    axes[2].plot(t_slow, X_slow[idx, :, 1], label="TEMP")
    axes[2].set_title("EDA/TEMP 4 Hz")
    axes[2].set_xlabel("Time (sec)")
    axes[2].set_ylabel("Norm. slow")
    axes[2].legend(loc="best")

    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()

# ============================================================
# Main pipeline
# ============================================================

def run_preprocessing(config: PreprocessingConfig) -> Dict[str, object]:
    paths = get_project_paths(config)
    create_project_dirs(paths)
    set_random_seed(config.random_seed)

    logger = setup_logging(paths["logs"] / "preprocessing.log")
    save_config_files(config, paths)

    logger.info("Starting common PPG-DaLiA native-rate preprocessing")
    logger.info("Project root: %s", config.project_root)
    logger.info("Raw data dir: %s", config.raw_data_dir)

    raw_dir = Path(config.raw_data_dir)
    if not raw_dir.exists():
        raise FileNotFoundError(f"Raw data directory does not exist: {raw_dir}")

    lengths = compute_window_lengths(config)
    logger.info("Window lengths: %s", lengths)

    X_acc_all_list = []
    X_bvp_all_list = []
    X_slow_all_list = []
    y_all_list = []
    subject_all_list = []
    coverage_all_list = []

    start_time_all_list = []
    end_time_all_list = []
    acc_start_all_list = []
    acc_end_all_list = []
    bvp_start_all_list = []
    bvp_end_all_list = []
    slow_start_all_list = []
    slow_end_all_list = []
    label_start_all_list = []
    label_end_all_list = []

    subject_summary_list = []

    for subject_id in config.subject_ids:
        result = process_subject(subject_id, config, logger)

        if result is None:
            continue

        X_acc_all_list.append(result["X_acc"])
        X_bvp_all_list.append(result["X_bvp"])
        X_slow_all_list.append(result["X_slow"])
        y_all_list.append(result["y"])
        subject_all_list.append(result["subjects"])
        coverage_all_list.append(result["coverage"])

        start_time_all_list.append(result["start_time_sec"])
        end_time_all_list.append(result["end_time_sec"])
        acc_start_all_list.append(result["acc_start"])
        acc_end_all_list.append(result["acc_end"])
        bvp_start_all_list.append(result["bvp_start"])
        bvp_end_all_list.append(result["bvp_end"])
        slow_start_all_list.append(result["slow_start"])
        slow_end_all_list.append(result["slow_end"])
        label_start_all_list.append(result["label_start"])
        label_end_all_list.append(result["label_end"])

        subject_summary_list.append(result["summary"])

    if len(X_acc_all_list) == 0:
        raise RuntimeError("No subjects were processed. Check raw_data_dir and folder structure.")

    X_acc_raw = np.concatenate(X_acc_all_list, axis=0).astype(np.float32)
    X_bvp_raw = np.concatenate(X_bvp_all_list, axis=0).astype(np.float32)
    X_slow_raw = np.concatenate(X_slow_all_list, axis=0).astype(np.float32)

    y_all = np.concatenate(y_all_list, axis=0).astype(np.int64)
    subjects_all = np.concatenate(subject_all_list, axis=0).astype(str)
    coverage_all = np.concatenate(coverage_all_list, axis=0).astype(np.float32)

    start_time_all = np.concatenate(start_time_all_list, axis=0).astype(np.float32)
    end_time_all = np.concatenate(end_time_all_list, axis=0).astype(np.float32)
    acc_start_all = np.concatenate(acc_start_all_list, axis=0).astype(np.int64)
    acc_end_all = np.concatenate(acc_end_all_list, axis=0).astype(np.int64)
    bvp_start_all = np.concatenate(bvp_start_all_list, axis=0).astype(np.int64)
    bvp_end_all = np.concatenate(bvp_end_all_list, axis=0).astype(np.int64)
    slow_start_all = np.concatenate(slow_start_all_list, axis=0).astype(np.int64)
    slow_end_all = np.concatenate(slow_end_all_list, axis=0).astype(np.int64)
    label_start_all = np.concatenate(label_start_all_list, axis=0).astype(np.int64)
    label_end_all = np.concatenate(label_end_all_list, axis=0).astype(np.int64)

    logger.info("Combined raw dataset:")
    logger.info("  X_acc_raw:  %s", X_acc_raw.shape)
    logger.info("  X_bvp_raw:  %s", X_bvp_raw.shape)
    logger.info("  X_slow_raw: %s", X_slow_raw.shape)
    logger.info("  y_all:      %s", y_all.shape)
    logger.info("  subjects:   %s", subjects_all.shape)

    # Group-compatible global normalization.
    acc_mean, acc_std = compute_mean_std(X_acc_raw)
    bvp_mean, bvp_std = compute_mean_std(X_bvp_raw)
    slow_mean, slow_std = compute_mean_std(X_slow_raw)

    X_acc_norm = normalize(X_acc_raw, acc_mean, acc_std)
    X_bvp_norm = normalize(X_bvp_raw, bvp_mean, bvp_std)
    X_slow_norm = normalize(X_slow_raw, slow_mean, slow_std)

    logger.info("Applied global normalization to match the shared group preprocessing format.")

    metadata_df = pd.DataFrame({
        "global_window_id": np.arange(len(y_all), dtype=np.int64),
        "subject": subjects_all,
        "activity_label": y_all,
        "activity_name": [config.activity_names.get(int(label), str(int(label))) for label in y_all],
        "dominant_label_coverage": coverage_all,
        "start_time_sec": start_time_all,
        "end_time_sec": end_time_all,
        "acc_start_sample_32hz": acc_start_all,
        "acc_end_sample_32hz": acc_end_all,
        "bvp_start_sample_64hz": bvp_start_all,
        "bvp_end_sample_64hz": bvp_end_all,
        "slow_start_sample_4hz": slow_start_all,
        "slow_end_sample_4hz": slow_end_all,
        "label_start_sample_4hz": label_start_all,
        "label_end_sample_4hz": label_end_all,
        "window_seconds": config.window_seconds,
        "shift_seconds": config.shift_seconds,
    })

    subject_summary_df = pd.DataFrame(subject_summary_list)

    # Save to both clean folder and compatibility folder.
    for output_dir in [paths["processed"]]:
        save_core_outputs(
            output_dir=output_dir,
            config=config,
            X_acc_raw=X_acc_raw,
            X_bvp_raw=X_bvp_raw,
            X_slow_raw=X_slow_raw,
            X_acc_norm=X_acc_norm,
            X_bvp_norm=X_bvp_norm,
            X_slow_norm=X_slow_norm,
            y_all=y_all,
            subjects_all=subjects_all,
            coverage_all=coverage_all,
            metadata_df=metadata_df,
            subject_summary_df=subject_summary_df,
            acc_mean=acc_mean,
            acc_std=acc_std,
            bvp_mean=bvp_mean,
            bvp_std=bvp_std,
            slow_mean=slow_mean,
            slow_std=slow_std,
        )

    if config.save_debug_csvs:
        save_debug_csvs(
            output_dir=paths["processed"],
            config=config,
            X_acc_norm=X_acc_norm,
            X_bvp_norm=X_bvp_norm,
            X_slow_norm=X_slow_norm,
            y=y_all,
            subjects=subjects_all,
            coverage=coverage_all,
            start_time_sec=start_time_all,
        )

    label_counts = {int(k): int(v) for k, v in zip(*np.unique(y_all, return_counts=True))}
    subject_counts = {str(k): int(v) for k, v in zip(*np.unique(subjects_all, return_counts=True))}

    norm_rows = []
    norm_rows.extend({"branch": "ACC", **row} for row in make_normalization_summary(
        config.acc_channel_names, acc_mean, acc_std, X_acc_norm
    ))
    norm_rows.extend({"branch": "BVP", **row} for row in make_normalization_summary(
        config.bvp_channel_names, bvp_mean, bvp_std, X_bvp_norm
    ))
    norm_rows.extend({"branch": "SLOW", **row} for row in make_normalization_summary(
        config.slow_channel_names, slow_mean, slow_std, X_slow_norm
    ))

    pd.DataFrame(norm_rows).to_csv(paths["results"] / "normalization_summary.csv", index=False)
    pd.DataFrame(
        [{"activity_label": k, "activity_name": config.activity_names.get(k, str(k)), "count": v}
         for k, v in sorted(label_counts.items())]
    ).to_csv(paths["results"] / "activity_distribution.csv", index=False)
    pd.DataFrame(
        [{"subject": k, "count": v}
         for k, v in sorted(subject_counts.items(), key=lambda item: int(item[0][1:]))]
    ).to_csv(paths["results"] / "subject_window_counts.csv", index=False)
    subject_summary_df.to_csv(paths["results"] / "subject_summary.csv", index=False)

    summary = {
        "num_windows": int(len(y_all)),
        "X_acc_shape": list(X_acc_norm.shape),
        "X_bvp_shape": list(X_bvp_norm.shape),
        "X_slow_shape": list(X_slow_norm.shape),
        "y_shape": list(y_all.shape),
        "subject_shape": list(subjects_all.shape),
        "label_counts": label_counts,
        "subject_counts": subject_counts,
        "clean_processed_folder": str(paths["processed"]),
        "normalization": "global normalization for compatibility with group preprocessing",
        "shared_split_protocol": {
            "train_subjects": config.train_subjects,
            "val_subjects": config.val_subjects,
            "test_subjects": config.test_subjects,
        },
    }

    save_json(summary, paths["results"] / "preprocessing_summary.json")

    plot_activity_distribution(
        y=y_all,
        config=config,
        figure_path=paths["figures"] / "activity_distribution.png",
    )
    plot_subject_window_counts(
        subjects=subjects_all,
        figure_path=paths["figures"] / "subject_window_counts.png",
    )
    plot_example_window(
        X_acc=X_acc_norm,
        X_bvp=X_bvp_norm,
        X_slow=X_slow_norm,
        y=y_all,
        subjects=subjects_all,
        config=config,
        figure_path=paths["figures"] / "example_native_rate_window.png",
    )

    write_discussion_markdown(
        path=paths["discussion"] / "01_preprocessing_discussion.md",
        config=config,
        summary=summary,
    )

    logger.info("Saved common preprocessing outputs.")
    logger.info("Clean processed folder: %s", paths["processed"])
    logger.info("Results folder: %s", paths["results"])
    logger.info("Figures folder: %s", paths["figures"])
    logger.info("Discussion file: %s", paths["discussion"] / "01_preprocessing_discussion.md")

    return {
        "summary": summary,
        "paths": {k: str(v) for k, v in paths.items()},
    }


# ============================================================
# Script entry point
# ============================================================

if __name__ == "__main__":
    outputs = run_preprocessing(CONFIG)
    print(json.dumps(outputs["summary"], indent=2))


2026-07-06 23:02:38 | INFO | Starting common PPG-DaLiA native-rate preprocessing
2026-07-06 23:02:38 | INFO | Project root: /home/iailab42/khans1/projects/ir
2026-07-06 23:02:38 | INFO | Raw data dir: /home/iailab42/g2-synthetic/PPG_FieldStudy
2026-07-06 23:02:38 | INFO | Window lengths: {'acc_window_len': 256, 'bvp_window_len': 512, 'slow_window_len': 32, 'label_window_len': 32, 'acc_shift_len': 64, 'bvp_shift_len': 128, 'slow_shift_len': 8, 'label_shift_len': 8}
2026-07-06 23:02:38 | INFO | ======================================================================
2026-07-06 23:02:38 | INFO | Processing S1
2026-07-06 23:02:38 | INFO | File: /home/iailab42/g2-synthetic/PPG_FieldStudy/S1/S1.pkl
2026-07-06 23:02:44 | INFO | Windowed S1:
2026-07-06 23:02:44 | INFO |   X_acc:  (3445, 256, 3)
2026-07-06 23:02:44 | INFO |   X_bvp:  (3445, 512, 1)
2026-07-06 23:02:44 | INFO |   X_slow: (3445, 32, 2)
2026-07-06 23:02:44 | INFO |   y:      (3445,)
2026-07-06 23:02:44 | INFO |   label counts: {np.i

{
  "num_windows": 46925,
  "X_acc_shape": [
    46925,
    256,
    3
  ],
  "X_bvp_shape": [
    46925,
    512,
    1
  ],
  "X_slow_shape": [
    46925,
    32,
    2
  ],
  "y_shape": [
    46925
  ],
  "subject_shape": [
    46925
  ],
  "label_counts": {
    "1": 4538,
    "2": 3206,
    "3": 2279,
    "4": 3442,
    "5": 6809,
    "6": 13520,
    "7": 4663,
    "8": 8468
  },
  "subject_counts": {
    "S1": 3445,
    "S10": 3532,
    "S11": 3490,
    "S12": 2944,
    "S13": 3356,
    "S14": 3177,
    "S15": 2923,
    "S2": 2823,
    "S3": 3344,
    "S4": 3295,
    "S5": 3345,
    "S6": 1463,
    "S7": 3553,
    "S8": 2978,
    "S9": 3257
  },
  "clean_processed_folder": "/home/iailab42/khans1/projects/ir/data/processed/native_rates",
  "normalization": "global normalization for compatibility with group preprocessing",
  "shared_split_protocol": {
    "train_subjects": [
      "S1",
      "S2",
      "S3",
      "S4",
      "S5",
      "S6",
      "S9",
      "S11",
      "S12",

## Run

Run the next cell. It will preprocess all available subjects and save the shared output files used by KoVAE/GAN/Diffusion and the evaluation notebooks.


In [2]:
outputs = run_preprocessing(CONFIG)
outputs['summary']

2026-07-06 23:04:04 | INFO | Starting common PPG-DaLiA native-rate preprocessing
2026-07-06 23:04:04 | INFO | Project root: /home/iailab42/khans1/projects/ir
2026-07-06 23:04:04 | INFO | Raw data dir: /home/iailab42/g2-synthetic/PPG_FieldStudy
2026-07-06 23:04:04 | INFO | Window lengths: {'acc_window_len': 256, 'bvp_window_len': 512, 'slow_window_len': 32, 'label_window_len': 32, 'acc_shift_len': 64, 'bvp_shift_len': 128, 'slow_shift_len': 8, 'label_shift_len': 8}
2026-07-06 23:04:04 | INFO | ======================================================================
2026-07-06 23:04:04 | INFO | Processing S1
2026-07-06 23:04:04 | INFO | File: /home/iailab42/g2-synthetic/PPG_FieldStudy/S1/S1.pkl
2026-07-06 23:04:08 | INFO | Windowed S1:
2026-07-06 23:04:08 | INFO |   X_acc:  (3445, 256, 3)
2026-07-06 23:04:08 | INFO |   X_bvp:  (3445, 512, 1)
2026-07-06 23:04:08 | INFO |   X_slow: (3445, 32, 2)
2026-07-06 23:04:08 | INFO |   y:      (3445,)
2026-07-06 23:04:08 | INFO |   label counts: {np.i

{'num_windows': 46925,
 'X_acc_shape': [46925, 256, 3],
 'X_bvp_shape': [46925, 512, 1],
 'X_slow_shape': [46925, 32, 2],
 'y_shape': [46925],
 'subject_shape': [46925],
 'label_counts': {1: 4538,
  2: 3206,
  3: 2279,
  4: 3442,
  5: 6809,
  6: 13520,
  7: 4663,
  8: 8468},
 'subject_counts': {'S1': 3445,
  'S10': 3532,
  'S11': 3490,
  'S12': 2944,
  'S13': 3356,
  'S14': 3177,
  'S15': 2923,
  'S2': 2823,
  'S3': 3344,
  'S4': 3295,
  'S5': 3345,
  'S6': 1463,
  'S7': 3553,
  'S8': 2978,
  'S9': 3257},
 'clean_processed_folder': '/home/iailab42/khans1/projects/ir/data/processed/native_rates',
 'normalization': 'global normalization for compatibility with group preprocessing',
 'shared_split_protocol': {'train_subjects': ['S1',
   'S2',
   'S3',
   'S4',
   'S5',
   'S6',
   'S9',
   'S11',
   'S12',
   'S13'],
  'val_subjects': ['S14', 'S15'],
  'test_subjects': ['S7', 'S8', 'S10']}}